# BAB 2 - Linear Regression
Regresi adalah bagian dari *supervised learning* di mana model dilatih menggunakan **data berlabel** untuk memprediksi **variabel target numerik** (misalnya harga, suhu, atau jumlah penjualan) berdasarkan variabel independen (*features*)

---

## A. Linear Regression
### 1. Simple Linear Regression
* **Konsep:** Memodelkan hubungan linear antara **1 fitur input** ($X$) dengan **1 target** ($y$)


In [15]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [8]:
# Simple vs Multiple Linear Regression

# Dataset
df = pd.DataFrame({
    'Pengalaman': [1, 2, 3, 4, 5, 6, 7, 8],
    'Sertifikasi': [1, 1, 2, 2, 3, 3, 4, 4],
    'Gaji': [4.5, 6.0, 8.5, 10.5, 13.0, 15.0, 18.0, 20.5]
})

# Simple Linear Regression
# Menggunakan 1 fitur (X) untuk memprediksi target (y)
# Digunakan saat ingin melihat pengaruh langsung dari 1 faktor utama
X_simple = df[["Pengalaman"]]
y = df["Gaji"]

model_simple = LinearRegression()
model_simple.fit(X_simple, y)
print("=== SIMPLE LINEAR REGRESSION ===")
print("Slope (Coeff) Pengalaman :", model_simple.coef_[0])
print("Intercept                :", model_simple.intercept_)
print("Prediksi 5 thn pengalaman:", model_simple.predict([[5]])[0])

# Multiple Linear Regression
# Menggunakan beberapa fitur (X) untuk memprediksi target (y)
X_multi = df[["Pengalaman", "Sertifikasi"]]

model_multi = LinearRegression()
model_multi.fit(X_multi, y)
print("\n=== MULTIPLE LINEAR REGRESSION ===")
print("Coefficients (Pengalaman, Sertifikasi):", model_multi.coef_)
print("Intercept                             :", model_multi.intercept_)
print("Prediksi 5 thn + 3 Sertifikasi        :", model_multi.predict([[5, 3]])[0])

=== SIMPLE LINEAR REGRESSION ===
Slope (Coeff) Pengalaman : 2.3095238095238098
Intercept                : 1.6071428571428559
Prediksi 5 thn pengalaman: 13.154761904761905

=== MULTIPLE LINEAR REGRESSION ===
Coefficients (Pengalaman, Sertifikasi): [2.   0.65]
Intercept                             : 1.3749999999999964
Prediksi 5 thn + 3 Sertifikasi        : 13.325000000000001


d:\course\python\machine-learning\06_machine_learning\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
d:\course\python\machine-learning\06_machine_learning\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [12]:
# Metriks Evaluasi Regresi

# Data Asli (y_true) dan Prediksi Model (y_pred)
y_true = np.array([4.5, 6.0, 8.5, 10.5, 13.0])
y_pred = np.array([4.8, 5.7, 8.9, 10.1, 13.2])

# MAE -> Mean Absolute Error
# Menjelaskan error yang meleset dengan satuan sama seperti data asli
mae = mean_absolute_error(y_true, y_pred)

# MSE -> Mean Squared Error
# Digunakan untuk fungsi matematika saat melatih model
mse = mean_squared_error(y_true, y_pred)

# RMSE -> Root MSE (akar dari MSE)
# Sangat sensitif terhadap outlier, sangat berguna jika outlier ingin dihukum secara berat
rmse = np.sqrt(mse)

# R2 -> R-Squared (satuan persen)
# Mengukur tingkat kebagusan korelasi keseluruhan model tanpa terikat suatu angka
r2 = r2_score(y_true, y_pred)

print(f"MAE  : {mae:.3f} Juta (Meleset rata-rata {mae*1000:.0f} ribu rupiah)")
print(f"MSE  : {mse:.3f}")
print(f"RMSE : {rmse:.3f} Juta (Meleset dengan hukuman kuadrat)")
print(f"R²   : {r2:.4f} (Model mampu menjelaskan {r2*100:.2f}% variasi data)")

MAE  : 0.320 Juta (Meleset rata-rata 320 ribu rupiah)
MSE  : 0.108
RMSE : 0.329 Juta (Meleset dengan hukuman kuadrat)
R²   : 0.9884 (Model mampu menjelaskan 98.84% variasi data)


In [36]:
# Dataset
# Set seed agar hasil acak selalu konsisten
np.random.seed(42)
n_samples = 1000

# Generating Data Sintetis Kompleks
luas = np.random.randint(40, 300, n_samples)
kamar = np.random.randint(1, 6, n_samples)
jarak = np.round(np.random.uniform(1.0, 25.0, n_samples), 1)
kondisi = np.random.choice(['Buruk', 'Cukup', 'Bagus', 'Mewah'], n_samples, p=[0.15, 0.4, 0.35, 0.1])
lokasi = np.random.choice(['Pinggiran', 'Suburban', 'Pusat Kota'], n_samples, p=[0.3, 0.5, 0.2])
kolam = np.random.choice(['Tidak', 'Ya'], n_samples, p=[0.7, 0.3])

# Rumus Logis Pembuatan Harga (Target) + Noise
harga = (
    luas * 2.5 + 
    kamar * 15 - 
    jarak * 8 + 
    pd.Series(kondisi).map({'Buruk': 0, 'Cukup': 30, 'Bagus': 70, 'Mewah': 150}) + 
    pd.Series(lokasi).map({'Pinggiran': 0, 'Suburban': 50, 'Pusat Kota': 120}) + 
    pd.Series(kolam).map({'Tidak': 0, 'Ya': 80}) + 
    np.random.normal(0, 25, n_samples) # Random noise
)

df_rumah = pd.DataFrame({
    'Luas_Bangunan': luas,
    'Jumlah_Kamar': kamar,
    'Jarak_Pusat_Kota': jarak,
    'Kondisi_Rumah': kondisi,
    'Lokasi': lokasi,
    'Ada_Kolam': kolam,
    'Harga': harga
})

df_rumah

,Luas_Bangunan,Jumlah_Kamar,Jarak_Pusat_Kota,Kondisi_Rumah,Lokasi,Ada_Kolam,Harga
0,142,3,24.0,Cukup,Pinggiran,Tidak,211.995157
1,146,3,12.7,Bagus,Suburban,Ya,557.029877
2,111,4,3.6,Bagus,Suburban,Tidak,383.172848
3,228,3,14.2,Buruk,Pusat Kota,Ya,686.458308
4,60,1,11.9,Buruk,Pinggiran,Tidak,79.055297
...,...,...,...,...,...,...,...
995,108,5,17.2,Mewah,Suburban,Ya,549.922512
996,289,5,7.2,Cukup,Suburban,Tidak,793.509013
997,75,5,9.3,Bagus,Suburban,Ya,379.257856
998,278,2,23.0,Cukup,Pusat Kota,Ya,740.189219


In [64]:
# Alur Kerja Lengkap

# ------------------------------------------------------------
# PISAHKAN FITUR DENGAN TARGET
# ------------------------------------------------------------
X = df_rumah.drop(columns=["Harga"])
y = df_rumah["Harga"]

# ------------------------------------------------------------
# SPLIT DATA (LATIH 80%, UJI 20%)
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

# ------------------------------------------------------------
# TRANSFORMASI FITUR ORDINAL (Kondisi Rumah)
# ------------------------------------------------------------
mapping_kondisi = OrdinalEncoder(categories=[["Buruk", "Cukup", "Bagus", "Mewah"]])
X_train_proc["Kondisi_Rumah"] = mapping_kondisi.fit_transform(X_train_proc[["Kondisi_Rumah"]])
X_test_proc["Kondisi_Rumah"] = mapping_kondisi.transform(X_test_proc[["Kondisi_Rumah"]])

# ------------------------------------------------------------
# TRANSFORMASI FITUR NOMINAL (Lokasi dan Kolam)
# ------------------------------------------------------------
ohe = OneHotEncoder(drop="first", sparse_output=False)
ohe_cols = ["Lokasi", "Ada_Kolam"]

# Transform Train
train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train_proc[ohe_cols]),
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_train_proc.index
)
X_train_proc = pd.concat([X_train_proc.drop(columns=ohe_cols), train_ohe], axis=1)

# Transform Test
test_ohe = pd.DataFrame(
    ohe.transform(X_test_proc[ohe_cols]),
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_test_proc.index
)
X_test_proc = pd.concat([X_test_proc.drop(columns=ohe_cols), test_ohe], axis=1)

# ------------------------------------------------------------
# FITUR SCALING (StandardScaler -> Fitur Numerik)
# ------------------------------------------------------------
scaler = StandardScaler()
num_cols = ["Luas_Bangunan", "Jumlah_Kamar", "Jarak_Pusat_Kota"]

X_train_proc[num_cols] = scaler.fit_transform(X_train_proc[num_cols])
X_test_proc[num_cols] = scaler.transform(X_test_proc[num_cols])

# ------------------------------------------------------------
# TRAINING MODEL (Linear Regression)
# ------------------------------------------------------------
linear_model = LinearRegression()
linear_model.fit(X_train_proc, y_train)

# ------------------------------------------------------------
# PREDIKSI DATA UJI
# ------------------------------------------------------------
y_pred = linear_model.predict(X_test_proc)

# ------------------------------------------------------------
# EVALUASI
# ------------------------------------------------------------
print("=== HASIL EVALUASI DENGAN TRAIN-TEST MANUAL ===")
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²   :", r2_score(y_test, y_pred))

=== HASIL EVALUASI DENGAN TRAIN-TEST MANUAL ===
RMSE : 25.298743514520698
R²   : 0.985814193676801
